# 🎬 관객 감정 몰입도 평가 시스템

영화 시사회/상영회에서 관객 표정을 분석해 **객관적 평론 점수**를 자동 산출합니다.

## 시스템 구조

```
[학습 단계]                     [추론 단계]
사진 7,000장                    리액션 영상 입력
    ↓                               ↓
Augmentation (4배 증강)         프레임 추출
    ↓                               ↓
YOLO + 감정 분류 모델 학습      얼굴 감지 + 감정 TOP3 분류
                                    ↓
                                몰입도 점수 계산
                                    ↓
                                평론 리포트 출력
```

## 사용 방법

각 셀을 위에서 아래로 순서대로 실행하세요:
1. **환경 설정** - 패키지 설치 및 경로 설정
2. **모듈 1: Trainer** - YOLO + 감정 분류 모델 학습 (사진 데이터)
3. **모듈 2: Detector** - 영상에서 얼굴 + 감정 추출
4. **모듈 3: Analyzer** - 몰입도 점수 계산
5. **모듈 4: Reporter** - 최종 평론 리포트 생성
6. **전체 실행** - 영상 1개 → 리포트까지 한 번에

## 1️⃣ 환경 설정

필요한 패키지를 설치하고 경로를 설정합니다.

In [ ]:
# 패키지 설치 (처음 한 번만)
!pip install ultralytics torch torchvision timm tqdm scikit-learn matplotlib seaborn pandas opencv-python-headless Pillow -q

print("✅ 패키지 설치 완료")

In [ ]:
# ═══════════════════════════════════════════════════
# 📁 경로 및 설정 (본인 환경에 맞게 수정)
# ═══════════════════════════════════════════════════

import os

# 데이터 경로
DATA_DIR     = "/workspace/user5/img_augmented"      # Augmentation 결과
OUTPUT_DIR   = "/workspace/user5/system_output"      # 모든 결과 저장
VIDEO_PATH   = "/workspace/user5/reaction.mp4"       # 분석할 영상

# 학습 설정 (A40 48GB 기준)
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
LR           = 1e-4
NUM_WORKERS  = 8
IMG_SIZE     = 224
FRAME_INTERVAL = 15   # 영상 분석 간격 (프레임)

# 감정 클래스
CLASSES     = ["anger", "blank", "happy", "panic", "sadness"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

# 감정별 몰입도 가중치 (xlsx 기준)
EMOTION_WEIGHTS = {
    "happy":   1.0,    # 긍정적 몰입
    "panic":   0.8,    # 강한 반응 (놀람/긴장)
    "sadness": 0.5,    # 감동
    "anger":   0.3,    # 부정적이나 반응 있음
    "blank":  -0.4,    # 무반응 = 몰입 낮음
}

# 출력 폴더 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 데이터 경로: {DATA_DIR}")
print(f"📁 출력 경로: {OUTPUT_DIR}")
print(f"📹 영상 경로: {VIDEO_PATH}")
print(f"🎭 감정 클래스: {CLASSES}")

In [ ]:
# GPU 확인
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

## 2️⃣ Trainer 모듈 - 감정 분류 모델 학습

EfficientNet-B7 + CBAM Attention으로 5감정 분류 모델을 파인튜닝합니다.
- 입력: 7,000장 + 증강 = 28,000장
- 출력: `best_emotion_model.pth`

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from PIL import Image

# ── Dataset ────────────────────────────────────────
class EmotionDataset(Dataset):
    def __init__(self, data_dir, split, transform=None):
        self.transform = transform
        self.samples = []
        split_dir = os.path.join(data_dir, split)
        for emotion in CLASSES:
            folder = os.path.join(split_dir, emotion)
            if not os.path.isdir(folder):
                continue
            label = CLASS_TO_IDX[emotion]
            for fname in os.listdir(folder):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

print("✅ Dataset 클래스 정의 완료")

In [ ]:
# ── CBAM Attention ─────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x).squeeze(-1).squeeze(-1))
        mx  = self.fc(self.max_pool(x).squeeze(-1).squeeze(-1))
        att = self.sigmoid(avg + mx).unsqueeze(-1).unsqueeze(-1)
        return x * att


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx  = torch.max(x, dim=1, keepdim=True)[0]
        att = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * att


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention()

    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


# ── EfficientNet-B7 + CBAM 모델 ───────────────────
class EmotionClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        import timm
        backbone = timm.create_model("efficientnet_b7", pretrained=True, num_classes=0)
        self.features = backbone
        feat_dim = backbone.num_features  # 2560

        self.cbam = CBAM(feat_dim, reduction=32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.bn = nn.BatchNorm1d(feat_dim)
        self.dropout = nn.Dropout(0.4)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        x = self.features.forward_features(x)
        x = self.cbam(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

print("✅ 모델 정의 완료 (EfficientNet-B7 + CBAM)")

In [ ]:
def train_emotion_model():
    """
    감정 분류 모델 학습 메인 함수
    """
    print("="*55)
    print("  감정 분류 모델 학습 시작")
    print("="*55)

    # 데이터 로드
    print("\n데이터 로드 중...")
    train_ds = EmotionDataset(DATA_DIR, "train", get_transforms(True))
    val_ds   = EmotionDataset(DATA_DIR, "val",   get_transforms(False))
    print(f"  Train: {len(train_ds)}장")
    print(f"  Val:   {len(val_ds)}장")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # 모델
    model = EmotionClassifier(NUM_CLASSES).to(DEVICE)
    print(f"  파라미터: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

    # 클래스 가중치 (불균형 보정)
    counts = [0] * NUM_CLASSES
    for _, l in train_ds.samples:
        counts[l] += 1
    weights = torch.tensor([max(counts)/c for c in counts], dtype=torch.float).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.cuda.amp.GradScaler()

    # 학습 루프
    save_path = os.path.join(OUTPUT_DIR, "best_emotion_model.pth")
    best_acc = 0.0
    history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}

    for epoch in range(1, NUM_EPOCHS + 1):
        # Train
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Train"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * imgs.size(0)
            train_correct += outputs.argmax(1).eq(labels).sum().item()
            train_total += imgs.size(0)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc="  Val", leave=False):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += outputs.argmax(1).eq(labels).sum().item()
                val_total += imgs.size(0)

        scheduler.step()

        train_loss /= train_total; train_acc = train_correct/train_total
        val_loss   /= val_total;   val_acc   = val_correct/val_total

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)

        print(f"\nEpoch {epoch}: Train Loss {train_loss:.4f} Acc {train_acc*100:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc*100:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "state_dict": model.state_dict(),
                "val_acc": val_acc,
                "classes": CLASSES
            }, save_path)
            print(f"  ✅ Best 저장 (val_acc: {best_acc*100:.2f}%)")

    print(f"\n🏆 최종 Best Val Acc: {best_acc*100:.2f}%")
    print(f"📁 저장 위치: {save_path}")
    return save_path, history

# ⚠️ 학습 시작 (시간 오래 걸림 - 데이터 양에 따라 1~6시간)
# 실행하려면 아래 주석 해제
# model_path, history = train_emotion_model()

## 3️⃣ Detector 모듈 - 영상에서 얼굴 + 감정 추출

YOLO-Face로 다중 얼굴 감지 + 학습한 감정 모델로 TOP3 추출

- 입력: 영상 파일 (mp4)
- 출력: 프레임별 감정 데이터 JSON

In [ ]:
import cv2
import numpy as np
import json
from datetime import datetime

class EmotionVideoDetector:
    """
    YOLO-Face + 감정 분류 통합 추론기
    """
    def __init__(self, emotion_model_path):
        # YOLO-Face 모델 로드
        from ultralytics import YOLO
        print("YOLO-Face 모델 로드 중...")
        self.yolo = YOLO("yolov8n-face.pt")  # 자동 다운로드

        # 감정 분류 모델 로드
        print("감정 분류 모델 로드 중...")
        self.emotion_model = EmotionClassifier(NUM_CLASSES).to(DEVICE)
        ckpt = torch.load(emotion_model_path, map_location=DEVICE)
        self.emotion_model.load_state_dict(ckpt["state_dict"])
        self.emotion_model.eval()

        # 감정 분류용 transform
        self.emotion_transform = get_transforms(is_train=False)

        print("✅ 모든 모델 로드 완료")

    def detect_faces(self, frame):
        """YOLO-Face로 얼굴 감지 → [(x,y,w,h,conf), ...]"""
        results = self.yolo(frame, verbose=False)
        faces = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                if conf > 0.3:
                    faces.append((x1, y1, x2-x1, y2-y1, conf))
        return faces

    def classify_emotion(self, face_img):
        """얼굴 이미지 → 감정 TOP3"""
        from PIL import Image
        face_pil = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
        tensor = self.emotion_transform(face_pil).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            with torch.cuda.amp.autocast():
                logits = self.emotion_model(tensor)
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        # TOP3 추출
        sorted_idx = np.argsort(probs)[::-1][:3]
        top3 = [
            {"emotion": IDX_TO_CLASS[i], "score": float(probs[i])}
            for i in sorted_idx
        ]
        return top3, {IDX_TO_CLASS[i]: float(probs[i]) for i in range(NUM_CLASSES)}

    def process_video(self, video_path, frame_interval=FRAME_INTERVAL):
        """영상 전체 분석 → 프레임별 결과 리스트"""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"영상 열기 실패: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps

        print(f"📹 영상: {duration:.1f}초 / {fps:.1f}fps / {total_frames}프레임")
        print(f"📊 분석 간격: {frame_interval}프레임 ({frame_interval/fps:.1f}초)")

        results = []
        frame_indices = list(range(0, total_frames, frame_interval))

        for frame_idx in tqdm(frame_indices, desc="영상 분석"):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                continue

            timestamp = frame_idx / fps
            faces = self.detect_faces(frame)

            frame_data = {
                "frame": frame_idx,
                "timestamp": round(timestamp, 2),
                "num_faces": len(faces),
                "faces": []
            }

            for (x, y, w, h, face_conf) in faces:
                face_img = frame[y:y+h, x:x+w]
                if face_img.size == 0:
                    continue
                try:
                    top3, all_scores = self.classify_emotion(face_img)
                    frame_data["faces"].append({
                        "bbox": [int(x), int(y), int(w), int(h)],
                        "face_confidence": round(face_conf, 3),
                        "top3": top3,
                        "all_scores": {k: round(v, 4) for k, v in all_scores.items()}
                    })
                except Exception as e:
                    pass

            results.append(frame_data)

        cap.release()
        return results, fps, duration

print("✅ EmotionVideoDetector 클래스 정의 완료")

## 4️⃣ Analyzer 모듈 - 몰입도 점수 계산

감정 데이터 → 3가지 요소로 몰입도 점수 산출:
- **감정 강도**: emotion_weight × confidence
- **표정 변화 빈도**: 감정이 자주 바뀔수록 ↑
- **화면 주시 비율**: 얼굴이 감지된 프레임 비율

In [ ]:
import pandas as pd

class EngagementAnalyzer:
    """
    감정 데이터 → 몰입도 점수 변환
    """
    def __init__(self, emotion_weights=None):
        self.weights = emotion_weights or EMOTION_WEIGHTS

    def calc_emotion_strength(self, top3):
        """감정 강도 = 가중치 × confidence"""
        if not top3:
            return 0
        top1 = top3[0]
        weight = self.weights.get(top1["emotion"], 0)
        return weight * top1["score"]

    def calc_change_rate(self, recent_emotions, window=10):
        """표정 변화 빈도 (이전 N개 프레임 대비 변화율)"""
        if len(recent_emotions) < 2:
            return 0
        changes = sum(1 for i in range(1, len(recent_emotions))
                      if recent_emotions[i] != recent_emotions[i-1])
        return changes / len(recent_emotions)

    def calc_attention_ratio(self, frame_data, total_frames):
        """화면 주시 비율 = 얼굴 감지된 프레임 / 전체 프레임"""
        face_detected = sum(1 for f in frame_data if f["num_faces"] > 0)
        return face_detected / max(total_frames, 1)

    def analyze(self, frame_results, fps, duration):
        """
        프레임 데이터 → 몰입도 타임라인 + 종합 점수
        """
        timeline = []
        recent_emotions = []  # 최근 감정 추적

        for frame in frame_results:
            ts = frame["timestamp"]

            if frame["num_faces"] == 0:
                # 얼굴 없으면 감정 강도 = 0
                emotion_strength = 0
                top1_emotion = None
                top1_conf = 0
            else:
                # 가장 큰 얼굴 기준 (배열 첫 번째)
                main_face = frame["faces"][0]
                top3 = main_face["top3"]
                emotion_strength = self.calc_emotion_strength(top3)
                top1_emotion = top3[0]["emotion"]
                top1_conf = top3[0]["score"]
                recent_emotions.append(top1_emotion)
                if len(recent_emotions) > 10:
                    recent_emotions.pop(0)

            change_rate = self.calc_change_rate(recent_emotions)

            # 단일 프레임 몰입도 (정규화: -1 ~ 1)
            engagement = emotion_strength + (change_rate * 0.3)

            timeline.append({
                "timestamp": ts,
                "frame": frame["frame"],
                "num_faces": frame["num_faces"],
                "top1_emotion": top1_emotion,
                "top1_confidence": top1_conf,
                "emotion_strength": round(emotion_strength, 4),
                "change_rate": round(change_rate, 4),
                "engagement": round(engagement, 4),
            })

        # 종합 점수
        attention = self.calc_attention_ratio(frame_results, len(frame_results))
        valid_engagements = [t["engagement"] for t in timeline if t["num_faces"] > 0]

        if not valid_engagements:
            avg_engagement = 0
        else:
            avg_engagement = sum(valid_engagements) / len(valid_engagements)

        # 최종 점수 (0~100 정규화)
        final_score = max(0, min(100, (avg_engagement + 1) * 50)) * attention

        # 구간별 통계
        df = pd.DataFrame(timeline)
        df["section"] = (df["timestamp"] // 10 * 10).astype(int)
        section_eng = df[df["num_faces"] > 0].groupby("section")["engagement"].mean()

        peak_section = section_eng.idxmax() if len(section_eng) > 0 else 0
        low_section  = section_eng.idxmin() if len(section_eng) > 0 else 0

        # 감정 분포
        valid_emotions = [t["top1_emotion"] for t in timeline if t["top1_emotion"]]
        from collections import Counter
        emotion_dist = dict(Counter(valid_emotions))

        return {
            "timeline":          timeline,
            "duration":          duration,
            "total_frames":      len(timeline),
            "attention_ratio":   round(attention, 3),
            "avg_engagement":    round(avg_engagement, 3),
            "final_score":       round(final_score, 1),
            "peak_section":      f"{int(peak_section)}~{int(peak_section)+10}초",
            "low_section":       f"{int(low_section)}~{int(low_section)+10}초",
            "emotion_dist":      emotion_dist,
            "section_scores":    {int(k): round(v, 3) for k, v in section_eng.items()},
        }

print("✅ EngagementAnalyzer 클래스 정의 완료")

## 5️⃣ Reporter 모듈 - 평론 리포트 생성

분석 결과를 그래프 + JSON + 요약 텍스트로 출력

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 감정별 색상
EMOTION_COLORS_VIZ = {
    "happy":   "#F9C74F",
    "panic":   "#9B5DE5",
    "sadness": "#4895EF",
    "anger":   "#F94144",
    "blank":   "#ADB5BD",
}


class EngagementReporter:
    """
    분석 결과 → 시각화 + 리포트
    """
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def generate(self, analysis_result, video_name="reaction"):
        # 1. 그래프 생성
        graph_path = self._plot_timeline(analysis_result, video_name)

        # 2. JSON 저장
        json_path = os.path.join(self.output_dir, f"{video_name}_analysis.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(analysis_result, f, indent=2, ensure_ascii=False)

        # 3. 텍스트 리포트
        report_text = self._generate_text_report(analysis_result, video_name)
        text_path = os.path.join(self.output_dir, f"{video_name}_report.txt")
        with open(text_path, "w", encoding="utf-8") as f:
            f.write(report_text)

        print(report_text)
        print(f"\n📁 결과 저장:")
        print(f"  📊 그래프: {graph_path}")
        print(f"  📄 JSON:   {json_path}")
        print(f"  📝 리포트: {text_path}")

        return {
            "graph": graph_path,
            "json": json_path,
            "text": text_path
        }

    def _plot_timeline(self, result, video_name):
        timeline = result["timeline"]
        df = pd.DataFrame(timeline)
        valid_df = df[df["num_faces"] > 0]

        fig, axes = plt.subplots(3, 1, figsize=(16, 12))
        fig.suptitle(f"🎬 영화 평론 분석 - {video_name}",
                     fontsize=16, fontweight="bold", y=0.995)

        # 1) 감정 변화 타임라인
        ax1 = axes[0]
        for emotion, color in EMOTION_COLORS_VIZ.items():
            sub = valid_df[valid_df["top1_emotion"] == emotion]
            if len(sub) > 0:
                ax1.scatter(sub["timestamp"], [emotion]*len(sub),
                           c=color, s=30, alpha=0.7, label=emotion)
        ax1.set_xlabel("시간 (초)")
        ax1.set_title("감정 변화 타임라인")
        ax1.legend(loc="upper right", ncol=5)
        ax1.grid(True, alpha=0.3)

        # 2) 몰입도 점수 + 이동평균
        ax2 = axes[1]
        eng_vals = valid_df["engagement"].values
        ts_vals = valid_df["timestamp"].values
        colors_eng = ["#F94144" if v < 0 else "#90BE6D" for v in eng_vals]
        ax2.bar(ts_vals, eng_vals, color=colors_eng, alpha=0.6, width=0.5)

        if len(eng_vals) >= 10:
            ma = pd.Series(eng_vals).rolling(window=10, center=True).mean()
            ax2.plot(ts_vals, ma, color="#212529", linewidth=2, label="이동평균(10)")
            ax2.legend()

        ax2.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
        ax2.set_xlabel("시간 (초)")
        ax2.set_ylabel("몰입도")
        ax2.set_title("몰입도 타임라인")
        ax2.grid(True, alpha=0.3)

        # 3) 감정 분포 파이차트
        ax3 = axes[2]
        emo_dist = result["emotion_dist"]
        if emo_dist:
            labels = list(emo_dist.keys())
            sizes = list(emo_dist.values())
            colors = [EMOTION_COLORS_VIZ.get(e, "#ADB5BD") for e in labels]
            ax3.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%",
                   startangle=90, pctdistance=0.8)
            ax3.set_title("전체 감정 분포")

        plt.tight_layout()
        save_path = os.path.join(self.output_dir, f"{video_name}_graph.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        return save_path

    def _generate_text_report(self, result, video_name):
        score = result["final_score"]

        # 점수 등급
        if score >= 80:
            grade = "🏆 매우 우수 (관객 강한 몰입)"
        elif score >= 60:
            grade = "👍 우수 (관객 잘 몰입)"
        elif score >= 40:
            grade = "📊 보통 (평이한 반응)"
        elif score >= 20:
            grade = "📉 미흡 (반응 약함)"
        else:
            grade = "❌ 부족 (관객 무관심)"

        # 주요 감정
        if result["emotion_dist"]:
            top_emotion = max(result["emotion_dist"], key=result["emotion_dist"].get)
            top_pct = result["emotion_dist"][top_emotion] / sum(result["emotion_dist"].values()) * 100
        else:
            top_emotion, top_pct = "없음", 0

        report = f"""
═══════════════════════════════════════════════════════
            🎬 영화 평론 분석 리포트
═══════════════════════════════════════════════════════
영상명:         {video_name}
분석 시간:      {result["duration"]:.1f}초 ({result["total_frames"]}프레임)

───────────────────────────────────────────────────────
📊 종합 점수: {score:.1f}/100  {grade}
───────────────────────────────────────────────────────

▶ 평균 몰입도:    {result["avg_engagement"]:+.3f}  (-1.0 ~ +1.0)
▶ 화면 주시율:    {result["attention_ratio"]*100:.1f}%
▶ 주요 감정:      {top_emotion} ({top_pct:.1f}%)

───────────────────────────────────────────────────────
🔥 반응 구간 분석
───────────────────────────────────────────────────────
▶ 최고 반응 구간: {result["peak_section"]}
▶ 최저 반응 구간: {result["low_section"]}

───────────────────────────────────────────────────────
🎭 감정 분포
───────────────────────────────────────────────────────"""
        for emotion, count in sorted(result["emotion_dist"].items(),
                                      key=lambda x: -x[1]):
            pct = count / sum(result["emotion_dist"].values()) * 100
            bar = "█" * int(pct / 2)
            report += f"\n  {emotion:<10}: {count:>4}회 ({pct:>5.1f}%) {bar}"

        report += "\n═══════════════════════════════════════════════════════\n"
        return report

print("✅ EngagementReporter 클래스 정의 완료")

## 6️⃣ 전체 파이프라인 실행

영상 1개 → 감정 분석 → 몰입도 점수 → 리포트까지 한 번에

In [ ]:
def run_full_pipeline(video_path, emotion_model_path):
    """
    영상 → 분석 → 리포트 한 번에 실행
    """
    print("="*55)
    print("  🎬 관객 몰입도 평가 시스템 실행")
    print("="*55)

    # 1. Detector
    print("\n[1/3] 영상 분석 시작")
    detector = EmotionVideoDetector(emotion_model_path)
    frame_results, fps, duration = detector.process_video(
        video_path, frame_interval=FRAME_INTERVAL
    )

    # 중간 데이터 저장
    raw_path = os.path.join(OUTPUT_DIR, "raw_detection.json")
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump({"fps": fps, "duration": duration, "frames": frame_results},
                  f, indent=2, ensure_ascii=False)
    print(f"  💾 원시 데이터: {raw_path}")

    # 2. Analyzer
    print("\n[2/3] 몰입도 점수 계산")
    analyzer = EngagementAnalyzer()
    analysis = analyzer.analyze(frame_results, fps, duration)

    # 3. Reporter
    print("\n[3/3] 리포트 생성")
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    reporter = EngagementReporter(OUTPUT_DIR)
    report_files = reporter.generate(analysis, video_name=video_name)

    print("\n✅ 전체 파이프라인 완료!")
    return analysis, report_files


# ═══════════════════════════════════════════════════
# 실행
# ═══════════════════════════════════════════════════
# emotion_model_path = os.path.join(OUTPUT_DIR, "best_emotion_model.pth")
# 
# if not os.path.exists(emotion_model_path):
#     print("⚠️  학습된 모델이 없어요. 먼저 train_emotion_model() 실행하세요.")
# elif not os.path.exists(VIDEO_PATH):
#     print(f"⚠️  영상이 없어요: {VIDEO_PATH}")
# else:
#     analysis, report_files = run_full_pipeline(VIDEO_PATH, emotion_model_path)

# ⚠️ 위 주석 해제하고 실행하세요

## 📋 실행 순서 요약

학습 ~ 리포트까지 단계별로 진행:

```python
# 1. (한 번만) 감정 분류 모델 학습
model_path, history = train_emotion_model()

# 2. 영상 분석 + 리포트 생성 (영상마다 반복)
analysis, report_files = run_full_pipeline(
    video_path="/workspace/user5/영화제목.mp4",
    emotion_model_path=os.path.join(OUTPUT_DIR, "best_emotion_model.pth")
)
```

## 🎯 결과물 위치

```
/workspace/user5/system_output/
├── best_emotion_model.pth          ← 학습된 모델
├── raw_detection.json              ← YOLO+감정 원시 데이터
├── 영상명_analysis.json            ← 분석 결과 JSON
├── 영상명_graph.png                ← 시각화 그래프
└── 영상명_report.txt               ← 텍스트 리포트
```

## 🔧 트러블슈팅

| 문제 | 해결 |
|------|------|
| YOLO 모델 다운로드 실패 | `pip install ultralytics --upgrade` |
| timm 에러 | `pip install timm --upgrade` |
| GPU 메모리 부족 | `BATCH_SIZE = 32`로 줄이기 |
| 영상 못 읽음 | `pip install opencv-python-headless` |